In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from databricks.sdk import WorkspaceClient
from lh import common, jobs

# jobs/01_deploy_jobs: create or update a daily job from config/jobs.yml.
# dev: run by whoever works in dev (Developer, Team Lead or Manager) from their own Git folder; it deploys THEIR job,
#      lh_<code>_dev_daily_<role>, which runs as them, on demand, and mails only them.
# test/prod: run by CI after a merge, or by the admin login (job runs as the environment's job account,
#      schedule paused until go-live).
# Failure-mail addresses are read from the secret scope lh_<env> (keys mail_<role>); they are never printed.
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
dbutils.widgets.dropdown("mode", "plan", ["plan", "apply"])
env, mode = dbutils.widgets.get("env"), dbutils.widgets.get("mode")
cfg = common.load_config(env)
root = os.path.dirname(os.getcwd())
jcfg = jobs.load_jobs(root)
role = jobs.my_role(jcfg, env, lambda g: spark.sql(f"SELECT is_account_group_member('{g}')").first()[0])

mail = {}
for r in jobs.mail_roles(jcfg, env, role):
    try:
        mail[r] = dbutils.secrets.get(f"lh_{env}", f"mail_{r}")
    except Exception:
        raise ValueError(f"scope lh_{env} has no key mail_{r}: run the setup-secret-scopes workflow first")

out = jobs.deploy(jobs.Api(WorkspaceClient()), jcfg, root, cfg["code"], env, apply=(mode == "apply"), mail=mail, role=role)
print(f"mode={mode}")
for k in ("job", "action", "job_id", "run_as", "schedule", "notify", "tasks"):
    print(f"{k:9} {out[k]}")
if mode == "apply" and env == "dev":
    # dev: the person's own audit row. test/prod are deployed by CI as sp_lh_deploy, which has no data access by
    # design (least privilege); that deploy is recorded by the GitHub deployment and the run ci_<code>_<env>_deploy_job.
    common.log_run(spark, cfg, "jobs_deploy", "success", 1, f"{out['job']} {out['action']}")